# Phase 1: Data Preparation & Environment Setup
This notebook handles loading, cleaning, and tokenizing English (FakeNewsNet) and Hindi (IFND) datasets for fake news detection.

In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer
import re
import string
import os

# Set up tokenizer
tokenizer = AutoTokenizer.from_pretrained('xlm-roberta-base')

## 1. Load Datasets

In [ ]:
# English Data
en_fake = pd.read_csv('data/english_fake.csv')
en_real = pd.read_csv('data/english_real.csv')
en_fake['label'] = 1
en_real['label'] = 0
en_df = pd.concat([en_fake, en_real])[['title', 'label']].rename(columns={'title': 'text'})

# Hindi Data
hi_df = pd.read_csv('data/hindi_dataset.csv', encoding='latin-1')
hi_df = hi_df.rename(columns={'Statement': 'text', 'Label': 'label'})
hi_df['label'] = hi_df['label'].apply(lambda x: 0 if x == 1 else 1) # Standardize: 1=Fake, 0=Real
hi_df = hi_df[['text', 'label']]

print(f"English samples: {len(en_df)}")
print(f"Hindi samples: {len(hi_df)}")

## 2. Advanced Cleaning

In [ ]:
def clean_text(text):
    if not isinstance(text, str): return ""
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = text.encode('ascii', 'ignore').decode('ascii')
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()

full_df = pd.concat([en_df, hi_df]).dropna()
full_df['text'] = full_df['text'].apply(clean_text)
print(f"Total cleaned samples: {len(full_df)}")

## 3. XLM-R Tokenization

In [ ]:
max_length = 128
tokens = tokenizer(
    list(full_df['text']),
    padding='max_length',
    truncation=True,
    max_length=max_length,
    return_tensors='pt'
)

print("Tokenization complete.")